# modeling

I just realized that BART update rule code was wrong. I need to do the model fitting first. 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm


In [2]:
outputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\old_modeling\param_recovery_1_model_data"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\old_modeling\param_recovery_0_data"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [3]:

matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)
rows = []

for pt in range(nPatients):
# for pt in range(5):
    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    nTrials = int(TDdataParamRecovery.nTrials)

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float)
    reward = np.asarray(TDdataParamRecovery.Reward, dtype=float)
    reward = reward[:nTrials]

    result_raw = np.asarray(TDdataParamRecovery.result, dtype=str)
    result_raw = result_raw[:nTrials]

    inverseTemperatureRSTD = np.full((len(alphas), len(alphas)), np.nan)
    fit_score_loglik = np.full((len(alphas), len(alphas)), np.nan)

    expectedReward_1d_all = np.full((len(alphas), len(alphas), nTrials), np.nan, dtype=float)
    RewardPE_1d_all = np.full((len(alphas), len(alphas), nTrials), np.nan, dtype=float)
    predictor_all = np.full((len(alphas), len(alphas), nTrials), np.nan, dtype=float)

    # banked = 1, popped = 0
    y = np.array([1 if x == "banked" else 0 for x in result_raw], dtype=float)

    for ap in range(len(alphas) - 1, -1, -1):
        for an in range(len(alphas) - 1, -1, -1):

            RewardPE = np.zeros(nTrials, dtype=float)
            expectedReward = np.zeros(nTrials, dtype=float)

            # predictor used in GLM: value BEFORE current trial outcome
            predictor = np.full(nTrials, np.nan, dtype=float)
            expectedReward_1d = np.full(nTrials, np.nan, dtype=float)
            RewardPE_1d = np.full(nTrials, np.nan, dtype=float)

            predictor[0] = expectedReward[0]
            expectedReward_1d[0] = expectedReward[0]
            RewardPE_1d[0] = RewardPE[0]

            for t in range(1, nTrials):
                # use pre-outcome expected value to predict current outcome
                predictor[t] = expectedReward[t - 1]

                RewardPE[t] = reward[t] - expectedReward[t - 1]

                if RewardPE[t] > 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[ap] * RewardPE[t]
                elif RewardPE[t] < 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[an] * RewardPE[t]
                else:
                    expectedReward[t] = expectedReward[t - 1]

                expectedReward_1d[t] = expectedReward[t]
                RewardPE_1d[t] = RewardPE[t]

            expectedReward_1d_all[ap, an, :] = expectedReward_1d
            RewardPE_1d_all[ap, an, :] = RewardPE_1d
            predictor_all[ap, an, :] = predictor

            # skip first trial because it has no real history
            X = sm.add_constant(predictor[1:])
            y_fit = y[1:]

            try:
                model = sm.GLM(
                    y_fit,
                    X,
                    family=sm.families.Binomial(link=sm.families.links.Logit())
                )
                fit_result = model.fit()

                inverseTemperatureRSTD[ap, an] = fit_result.params[1]
                fit_score_loglik[ap, an] = fit_result.llf

            except:
                continue

    # choose best alpha pair from log-likelihood
    best_idx = np.unravel_index(np.nanargmax(fit_score_loglik), fit_score_loglik.shape)
    bestAlphaPosIdx, bestAlphaNegIdx = best_idx

    bestAlphaPos = alphas[bestAlphaPosIdx]
    bestAlphaNeg = alphas[bestAlphaNegIdx]

    bestExpectedReward = expectedReward_1d_all[bestAlphaPosIdx, bestAlphaNegIdx, :]
    bestRewardPE = RewardPE_1d_all[bestAlphaPosIdx, bestAlphaNegIdx, :]
    bestPredictor = predictor_all[bestAlphaPosIdx, bestAlphaNegIdx, :]
    bestInverseTemperatureRSTD = inverseTemperatureRSTD[bestAlphaPosIdx, bestAlphaNegIdx]
    bestFitScoreLogLik = fit_score_loglik[bestAlphaPosIdx, bestAlphaNegIdx]


    td_dict = {}
    for key in TDdataParamRecovery._fieldnames:
        if key not in [
            "bestAlphaPos",
            "bestAlphaNeg",
            "bestExpectedReward",
            "bestRewardPE",
            "bestInverseTemperatureRSTD",
            "bestFitScoreLogLik",
            "inverseTemperatureRSTD"
        ]:
            td_dict[key] = getattr(TDdataParamRecovery, key)

    td_dict["bestAlphaPos"] = bestAlphaPos
    td_dict["bestAlphaNeg"] = bestAlphaNeg
    td_dict["bestExpectedReward"] = bestExpectedReward
    td_dict["bestRewardPE"] = bestRewardPE
    td_dict["bestInverseTemperatureRSTD"] = bestInverseTemperatureRSTD
    td_dict["bestFitScoreLogLik"] = bestFitScoreLogLik
    td_dict["inverseTemperatureRSTD"] = inverseTemperatureRSTD


    save_file_name = f"{ptID}_TDdataParamRecovery.mat"
    save_path = os.path.join(outputFolderName, save_file_name)

    savemat(save_path, {"TDdataParamRecovery": td_dict})

processing pt 1/71: 201810
processing pt 2/71: 201811
processing pt 3/71: 201901
processing pt 4/71: 201902r
processing pt 5/71: 201902
processing pt 6/71: 201903
processing pt 7/71: 201905
processing pt 8/71: 201909
processing pt 9/71: 201910
processing pt 10/71: 201911
processing pt 11/71: 201913
processing pt 12/71: 201914
processing pt 13/71: 201915
processing pt 14/71: 202001
processing pt 15/71: 202002
processing pt 16/71: 202003
processing pt 17/71: 202004
processing pt 18/71: 202005
processing pt 19/71: 202006u
processing pt 20/71: 202006
processing pt 21/71: 202007
processing pt 22/71: 202008
processing pt 23/71: 202009
processing pt 24/71: 202011
processing pt 25/71: 202014
processing pt 26/71: 202015
processing pt 27/71: 202016
processing pt 28/71: 202105
processing pt 29/71: 202107
processing pt 30/71: 202110
processing pt 31/71: 202114
processing pt 32/71: 202117
processing pt 33/71: 202118
processing pt 34/71: 202201
processing pt 35/71: 202202
processing pt 36/71: 202205

# debug

In [4]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['Reward', 'a', 'inflate_time', 'is_control', 'nTrials', 'points', 'pointsMinusReward', 'result', 'trial_type']


In [5]:
expectedReward.shape

(231,)